# Faruq-v3 — IGEM1 → AF2 Targeted Rescue Audit

Validation-only, post-training, **tanpa inference ulang dan tanpa training**. Audit memakai event JSON seed42 yang sudah ada. Tujuan: menguji apakah rescue AF2 terhadap error IGEM1 terkonsentrasi pada kelas/confusion family tertentu. Test tidak dibuka.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/multimodel-complementarity-audit'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())
print('coffee_detector import: OK')

In [ ]:
BASE = 'experiments/faruq-v3-multimodel-complementarity-seed42-v1/events'
IGEM_REL = f'{BASE}/IGEM1_seed42_events.json'
AF2_REL = f'{BASE}/AF2_seed42_events.json'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(IGEM_REL, AF2_REL))
IGEM = require_project_artifact(PROJECT_ROOT, IGEM_REL)
AF2 = require_project_artifact(PROJECT_ROOT, AF2_REL)
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-igem-af2-targeted-rescue-seed42-v1/igem_af2_targeted_rescue.json'
print('IGEM:', IGEM)
print('AF2 :', AF2)
print('OUT :', OUTPUT)

In [ ]:
command = [
    sys.executable, '-m', 'coffee_detector.analysis.igem_af2_targeted_rescue_audit',
    '--igem', str(IGEM), '--af2', str(AF2), '--output', str(OUTPUT),
]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
print('AUDIT SELESAI:', OUTPUT)

In [ ]:
import pandas as pd
from IPython.display import display

print('GLOBAL')
display(pd.DataFrame([result['global']]).style.format({
    'total_error_rescue_rate':'{:.2%}',
    'total_error_rescue_wilson95_low':'{:.2%}',
    'total_error_rescue_wilson95_high':'{:.2%}',
    'classification_rescue_rate':'{:.2%}',
    'classification_rescue_wilson95_low':'{:.2%}',
    'classification_rescue_wilson95_high':'{:.2%}',
}))

print('RESCUE CONCENTRATION')
for name, row in result['rescue_concentration'].items():
    print(name, row)

print('PER CLASS — sorted by classification rescues, then support')
class_rows = []
for cls, row in result['per_class'].items():
    class_rows.append({'class': cls, **row})
class_df = pd.DataFrame(class_rows).sort_values(
    ['af2_classification_rescues_iou50','igem_classification_errors_iou50','classification_rescue_rate'],
    ascending=False
)
display(class_df.style.format({
    'total_error_rescue_rate':'{:.2%}',
    'total_error_rescue_wilson95_low':'{:.2%}',
    'total_error_rescue_wilson95_high':'{:.2%}',
    'classification_rescue_rate':'{:.2%}',
    'classification_rescue_wilson95_low':'{:.2%}',
    'classification_rescue_wilson95_high':'{:.2%}',
}))

print('DIRECTED CONFUSION FAMILIES — GT -> IGEM wrong prediction')
directed = pd.DataFrame(result['directed_confusion_families'])
display(directed.head(25).style.format({'rescue_rate':'{:.2%}','wilson95_low':'{:.2%}','wilson95_high':'{:.2%}'}))

print('UNDIRECTED CLASS-PAIR FAMILIES')
undirected = pd.DataFrame(result['undirected_confusion_families'])
display(undirected.head(25).style.format({'rescue_rate':'{:.2%}','wilson95_low':'{:.2%}','wilson95_high':'{:.2%}'}))

print('Kirim GLOBAL + RESCUE CONCENTRATION + PER CLASS + top directed/undirected families. Jangan membuka test.')